In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}



def export_figure(figure, figure_id, tight=False):
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
from matplotlib.ticker import FixedLocator,FuncFormatter,NullLocator,NullFormatter
COLORS = ["#0072B2", "#E69F00", "#D55E00", "#666666"]
STYLES = ["-", "--", "-.", ":"]
PANEL_CODES = [[0, 1, 4, 7, 13, 10], [0, 2, 5, 8, 14, 11], [0, 3, 6, 9, 15, 12]]
TITLES = ["Noiseless", "Encoder Pauli-Y", "Encoder isotropic Pauli",
          "Encoder amplitude damping", "CNOT amplitude damping", "CNOT thermal relaxation"]
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9.5,
    "axes.titlesize": 9.6, "axes.labelsize": 9.6, "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5, "legend.fontsize": 8.6, "axes.linewidth": .75,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.facecolor": "white"})
def subtitle(panel, row):
    if panel == 0:
        return "shared ideal baseline"
    if panel in (1, 2):
        return r"$q_{\mathrm{total}}=" + f'{float(row["strength_value"]):g}' + "$"
    if panel == 3:
        return (r"$q_{\mathrm{ref}}=" + f'{float(row["strength_value"]):g}' +
                r",\ \gamma=" + f'{float(row["gamma_AD"]):.5f}' + "$")
    return (r"$\tau=" + f'{float(row["strength_value"]):g}' +
            (r",\ T_2=2T_1$" if panel == 4 else r",\ T_2=T_1$"))

In [ ]:
data=load_npz("data/FigS7/rebinned_errors.npz")
configurations=list(csv.DictReader((PACKAGE_ROOT/"data/FigS7/configurations.csv").open()))
ratio=data["ratio"]
median,lower,upper=np.median(ratio,axis=1),ratio.min(axis=1),ratio.max(axis=1)
ymin=float(ratio.min()); ymax=float(ratio.max())
log_floor=math.floor(ymin/.05)*.05
if log_floor<=0:log_floor=ymin/1.2
ytop=round(max(1.05,math.ceil((ymax+.02)/.05)*.05),10)
assert 0<log_floor<ymin and ytop>ymax

COLORS=["#171717"]+COLORS
STYLES=[(0,(8,2))]+STYLES
view="G1G4";bands=[1,2,3,4];scale="log";level=2;indices=PANEL_CODES[1]

In [ ]:
handles=[Line2D([],[],color=COLORS[b],ls=STYLES[b],lw=1.6,
          label=f"G{b}: "+r"$"+f"{data['edges'][b]:g}<r\\leq{data['edges'][b+1]:g}"+"$") for b in bands]
legend_handles=handles if len(bands)==4 else [handles[i] for i in [0,3,1,4,2]]

fig,axes=plt.subplots(2,3,figsize=(8.8,6.5 if len(bands)==4 else 6.8),sharex=True,sharey=True)
fig.subplots_adjust(left=.095,right=.985,bottom=.20,top=.80 if len(bands)==4 else .76,
                    wspace=.22,hspace=.50)
fig.legend(handles=legend_handles,loc="upper center",bbox_to_anchor=(.5,.935),
           ncol=4 if len(bands)==4 else 3,frameon=False,columnspacing=1.4,handlelength=2.5)
for panel,(ax,i) in enumerate(zip(axes.flat,indices)):
    for b in bands:
        line,=ax.plot(data["updates"],median[i,:,b],color=COLORS[b],ls=STYLES[b],lw=1.6)
        ax.fill_between(data["updates"],lower[i,:,b],upper[i,:,b],color=COLORS[b],alpha=.12,linewidth=0)
        np.testing.assert_array_equal(line.get_ydata(),median[i,:,b])
    ax.axhline(1,color=".60",lw=.65,ls=(0,(2,3)),zorder=0)
    ax.axhline(.5,color=".65",lw=.65,ls=(0,(5,3)),zorder=0)
    ax.set_title(f"({chr(97+panel)}) {TITLES[panel]}\n{subtitle(panel,configurations[i])}",pad=8)
    ax.set_xlim(0,500);ax.set_xticks([0,125,250,375,500]);ax.set_yscale(scale)
    ax.set_ylim((0,ytop) if scale=="linear" else (log_floor,ytop))
    if scale=="linear":ax.set_yticks([0,.25,.5,.75,1])
    else:
        ticks=[x for x in [.1,.2,.3,.4,.5,.6,.8,1,1.5,2] if log_floor<=x<=ytop]
        ax.yaxis.set_major_locator(FixedLocator(ticks))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v,p:f"{v:g}"))
        ax.yaxis.set_minor_locator(NullLocator());ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(axis="y",alpha=.16,lw=.55)
    ax.spines["top"].set_visible(False);ax.spines["right"].set_visible(False)
for ax in axes[1]:ax.set_xlabel("GD update")
for ax in axes[:,0]:ax.set_ylabel(r"Relative radial error $R_G(n)$")

export_figure(fig,"FigS7")
plt.close(fig)